In [1]:
"""
🟢 A-1. 하이퍼파라미터를 바꿔 가며 관찰하기 (가벼움 · 목표 40분)

[이 파일은 그대로 실행된다.] 고칠 곳은 아래 CONFIG 딱 하나다.

    python missions/A1_hparams.py

[하는 일]
CONFIG의 값을 **한 번에 하나씩만** 바꾸고 다시 실행해, 정확도·손실이 어떻게 달라지는지 본다.
한 번에 두 개를 바꾸면 무엇 때문에 달라졌는지 알 수 없다. (어제도 같은 원칙)

⚠️ 훈련은 실행마다 값이 조금씩 흔들린다. 그러니 **SEED와 설정을 함께 기록**해야
   비교가 의미가 있다. 맨 아래 '관찰 기록' 칸에 한 줄씩 적자. 회고에서 함께 본다.

[예상하고 나서 확인하기]
값을 바꾸기 전에 "이렇게 하면 정확도가 오를까/내릴까"를 먼저 예상하고 실행하자.
특히 MAX_LEN은 예상이 빗나가기 쉽다(왜 그런지는 06_why_long_fails.py).
"""

import sys
import time
from pathlib import Path

try:
    _HERE = Path(__file__).resolve().parent
except NameError:      # 노트북 셀에는 __file__ 이 없다
    _HERE = Path.cwd()
sys.path.insert(0, str(_HERE.parent))

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

from imdb_data import load_imdb
from textutils import tokenize_en, build_vocab, pad_and_tensor
from model import IMDBRnn

In [2]:
CONFIG = {
    "SEED": 42,          # 바꾸면 결과가 흔들린다. 비교할 땐 고정하고 하나씩만 바꾸자
    "VOCAB_SIZE": 2000,  # 사전 크기. 500 → 1000 → 2000
    "MAX_LEN": 100,      # 문장 길이. 50 → 100 → 200  (예상해 보고!)
    "HIDDEN": 32,        # 은닉 차원(기억 용량). 8 → 16 → 32
    "LR": 1e-3,          # 학습률. 1e-4 → 2e-4 → 1e-3
    "EPOCHS": 12,        # 에포크 수
}

In [3]:
c = CONFIG
torch.manual_seed(c["SEED"])

train, val = load_imdb(n_train=5000, n_val=2000)
train_toks = [tokenize_en(t) for t in train["text"]]
val_toks = [tokenize_en(t) for t in val["text"]]
word2idx, _ = build_vocab(train_toks, max_size=c["VOCAB_SIZE"])

X_train = pad_and_tensor(train_toks, word2idx, c["MAX_LEN"])
y_train = torch.tensor(train["label"], dtype=torch.float32)
X_val = pad_and_tensor(val_toks, word2idx, c["MAX_LEN"])
y_val = torch.tensor(val["label"], dtype=torch.float32)
tl = DataLoader(TensorDataset(X_train, y_train), batch_size=64, shuffle=True)
vl = DataLoader(TensorDataset(X_val, y_val), batch_size=64)

model = IMDBRnn(len(word2idx), embed_dim=32, hidden_size=c["HIDDEN"])
crit = nn.BCELoss()
opt = torch.optim.Adam(model.parameters(), lr=c["LR"])

print(f"설정: {c}")
best = 0.0
for ep in range(c["EPOCHS"]):
    t0 = time.time()
    model.train()
    for xb, yb in tl:
        opt.zero_grad(); crit(model(xb), yb).backward(); opt.step()
    model.eval(); cor = n = 0
    with torch.no_grad():
        for xb, yb in vl:
            cor += ((model(xb) > 0.5).float() == yb).sum().item(); n += len(yb)
    acc = cor / n; best = max(best, acc)
    print(f"  에포크 {ep+1:2d}: 검증정확도 {acc:.4f}  ({time.time()-t0:.1f}s)")

print(f"\n★ 최고 검증 정확도: {best:.4f}  (설정: VOCAB={c['VOCAB_SIZE']} "
      f"MAX_LEN={c['MAX_LEN']} HIDDEN={c['HIDDEN']} LR={c['LR']} SEED={c['SEED']})")

설정: {'SEED': 42, 'VOCAB_SIZE': 2000, 'MAX_LEN': 100, 'HIDDEN': 32, 'LR': 0.001, 'EPOCHS': 12}
  에포크  1: 검증정확도 0.5195  (1.2s)
  에포크  2: 검증정확도 0.5250  (1.4s)
  에포크  3: 검증정확도 0.5330  (1.3s)
  에포크  4: 검증정확도 0.5635  (1.1s)
  에포크  5: 검증정확도 0.5900  (1.0s)
  에포크  6: 검증정확도 0.6045  (1.0s)
  에포크  7: 검증정확도 0.5985  (1.0s)
  에포크  8: 검증정확도 0.6420  (0.9s)
  에포크  9: 검증정확도 0.6815  (1.0s)
  에포크 10: 검증정확도 0.6190  (1.2s)
  에포크 11: 검증정확도 0.6875  (1.3s)
  에포크 12: 검증정확도 0.6860  (1.0s)

★ 최고 검증 정확도: 0.6875  (설정: VOCAB=2000 MAX_LEN=100 HIDDEN=32 LR=0.001 SEED=42)


## 관찰 기록 (한 줄씩 적자 — 설정을 꼭 같이)

예)  MAX_LEN 50 → 0.70 / 100 → 0.68 / 200 → 0.69 : 길게 해도 안 오른다. 왜? (06 참고)

- HIDDEN  8 → __ / 16 → __ / 32 → __ :
- MAX_LEN 50 → __ / 100 → __ / 200 → __ :
- LR   1e-4 → __ / 2e-4 → __ / 1e-3 → __ :
- VOCAB 500 → __ / 1000 → __ / 2000 → __ :

한 문장 결론: